In [ ]:
%pip install kafka-python faker

In [ ]:
from kafka import KafkaConsumer, KafkaProducer
import json
import random

consumer = KafkaConsumer(
    'ride_assignments',
    bootstrap_servers='kafka-3ad4f62-project-746a.c.aivencloud.com:24264',
    security_protocol="SSL",
    ssl_cafile="/content/ca.pem",
    ssl_certfile="/content/service.cert",
    ssl_keyfile="/content/service.key",
    ssl_check_hostname=False,
    auto_offset_reset='earliest',
    enable_auto_commit=True,
    group_id='app-service-group',
    value_deserializer=lambda m: json.loads(m.decode('utf-8')),
    request_timeout_ms=30000,   # tiempo de espera más largo
    session_timeout_ms=29000    # evita desconexiones rápidas
)

producer = KafkaProducer(
    bootstrap_servers='kafka-3ad4f62-project-746a.c.aivencloud.com:24264',
    security_protocol="SSL",
    ssl_cafile="/content/ca.pem",
    ssl_certfile="/content/service.cert",
    ssl_keyfile="/content/service.key",
    value_serializer=lambda v: json.dumps(v).encode('utf-8')
)
count = 0
for msg in consumer:
    ride = msg.value

    process = {
        "ride_id": ride["ride_id"],
        "driver_id": ride["driver_id"],
        "status": "STARTED"
    }

    producer.send("ride_events", process)
    print("STARTED:", process)
    count += 1
    if count == 10:
        break

In [ ]:
from kafka import KafkaConsumer, KafkaProducer
import json
import random

consumer = KafkaConsumer(
    'ride_events',
    bootstrap_servers='kafka-3ad4f62-project-746a.c.aivencloud.com:24264',
    security_protocol="SSL",
    ssl_cafile="/content/ca.pem",
    ssl_certfile="/content/service.cert",
    ssl_keyfile="/content/service.key",
    ssl_check_hostname=False,
    auto_offset_reset='earliest',
    enable_auto_commit=True,
    group_id='app-service-group',
    value_deserializer=lambda m: json.loads(m.decode('utf-8')),
    request_timeout_ms=30000,   # tiempo de espera más largo
    session_timeout_ms=29000    # evita desconexiones rápidas
)

producer = KafkaProducer(
    bootstrap_servers='kafka-3ad4f62-project-746a.c.aivencloud.com:24264',
    security_protocol="SSL",
    ssl_cafile="/content/ca.pem",
    ssl_certfile="/content/service.cert",
    ssl_keyfile="/content/service.key",
    value_serializer=lambda v: json.dumps(v).encode('utf-8')
)
count = 0
for msg in consumer:
    ride = msg.value
    if ride["status"] == "STARTED":
      process = {
          "ride_id": ride["ride_id"],
          "driver_id": ride["driver_id"],
          "status": "COMPLETED"
      }

      producer.send("ride_events", process)
      print("COMPLETED:", process)
      count += 1
      if count == 10:
          break